# IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
# import tensorflow_hub as hub

# LOAD AND PREPARE DATA

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(df['text'])}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 5572
Average words per message: 16
Approximate vocabulary size: 15686


In [ ]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4457,), (1115,), (4457,), (1115,))

# MODEL DEVELOPMENT

## Baseline Model

In [ ]:
tfidfv = TfidfVectorizer()
X_train_tfidfv = tfidfv.fit_transform(X_train)
X_test_tfidfv = tfidfv.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidfv, y_train)

MultinomialNB()

In [ ]:
y_pred = model.predict(X_test_tfidfv)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.9632286995515695


In [36]:
messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

messages_tfidf = tfidfv.transform(messages)
predictions = model.predict(messages_tfidf)
# probabilities = model.predict_proba(new_message_tfidf)

for msg, pred in zip(messages, predictions):
    label = "Spam ❌" if pred == 1 else "Ham ✅ "
    print(f"{msg} --> {label}")


Hey, are we meeting today? --> Ham ✅ 
WIN cash now!!! Click the link --> Spam ❌
Your OTP is 348921 --> Ham ✅ 
You have won $1000! --> Spam ❌
Congratulations! You have won a free lottery ticket --> Ham ✅ 
Don't forget to submit the assignment --> Ham ✅ 
URGENT! You won a cash prize. Call immediately! --> Spam ❌
My friend won a prize yesterday --> Ham ✅ 
